# Tmap API 잔여 사용량 점검 (호출 1건)

`avoidZoneTest_v6` 노트북과 동일한 인증·엔드포인트 설정으로 보행자 경로 안내 API를 1회 호출하여
다음을 확인한다.

- 인증키 유효성 (HTTP 200 정상 / 401·403 인증 실패)
- 사용량 한도 소진 여부 (HTTP 429, 본문 `code: GW.QUOTA_LIMIT`)
- 게이트웨이가 응답 헤더로 노출하는 사용량 관련 필드(`*-quota-remaining`, `*-quota-limit` 등)

**주의**

- 잔여 호출 건수의 정확한 수치는 응답 본문에 표준 필드로 포함되지 않는다. 게이트웨이가 헤더로 노출하는
  경우에만 코드로 확인 가능하며, 그렇지 않으면 SK Open API 콘솔의 사용량 화면에서 확인해야 한다.
  (콘솔 경로: 홈 > 대시보드 > 요금 조회 > 사용상품관리)
- 이 노트북은 실행할 때마다 호출을 **정확히 1건** 소모한다. 반복 실행 시 그만큼 사용량이 추가로 차감된다.


In [ ]:
# 셀 1: 라이브러리 import + 인증키 로드
# avoidZoneTest_v6 셀 1·2와 동일하게 .env에서 TMAP_APP_KEY를 로드한다.

import os
import json
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

TMAP_APP_KEY = os.getenv("TMAP_APP_KEY")
if not TMAP_APP_KEY:
    raise RuntimeError(
        ".env에 TMAP_APP_KEY가 없다. avoidZoneTest_v6 노트북과 동일한 위치(.env)에 "
        "키가 존재하는지 확인한다."
    )

print("[인증키 로드]")
print(f"  Tmap: {TMAP_APP_KEY[:6]}...{TMAP_APP_KEY[-4:]} ({len(TMAP_APP_KEY)}자)")
print(f"  작업 디렉토리: {os.getcwd()}")


In [ ]:
# 셀 2: 보행자 경로 안내 API 1회 호출 + 사용량 점검
# 원본 노트북과 동일한 엔드포인트·페이로드 규격을 사용한다.
# 호출은 정확히 1건이며, 셀을 재실행하면 1건씩 추가로 소모된다.

TMAP_PEDESTRIAN_URL = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"

# 경로 유효성이 안정적인 짧은 보행 구간 1건 (강남역 -> 역삼역, 약 0.9km)
# 좌표는 점검 목적일 뿐이며, 프로젝트 대상 지역(인천) 좌표로 바꿔도 무방하다.
_START = (37.497942, 127.027621)   # (lat, lon)
_END   = (37.500622, 127.036456)

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "appKey": TMAP_APP_KEY,
}
payload = {
    "startX": _START[1], "startY": _START[0],
    "endX": _END[1], "endY": _END[0],
    "startName": "출발지", "endName": "도착지",
    "reqCoordType": "WGS84GEO", "resCoordType": "WGS84GEO",
    "searchOption": "0",
}

try:
    resp = requests.post(TMAP_PEDESTRIAN_URL, headers=headers, json=payload, timeout=15)
except requests.exceptions.RequestException as e:
    raise RuntimeError(f"네트워크 오류로 호출 실패: {type(e).__name__}: {e}")

print("=" * 64)
print(f"[HTTP 상태] {resp.status_code}")

# --- 1. 게이트웨이 사용량 관련 헤더 스캔 ---
# 응답 헤더 중 quota/limit/ratelimit/remaining 키워드를 포함하는 항목만 추출한다.
# 게이트웨이가 잔여량을 헤더로 노출하면 여기서 실제 남은 건수를 확인할 수 있다.
_KEYWORDS = ("quota", "ratelimit", "rate-limit", "remaining", "limit")
_quota_headers = {k: v for k, v in resp.headers.items()
                  if any(t in k.lower() for t in _KEYWORDS)}

print("\n[사용량 관련 응답 헤더]")
if _quota_headers:
    for k, v in _quota_headers.items():
        print(f"  {k}: {v}")
else:
    print("  없음 (이 게이트웨이는 사용량 정보를 응답 헤더로 노출하지 않음)")
    print("  -> 정확한 잔여 건수는 SK Open API 콘솔의 사용량 화면에서 확인한다.")

# --- 2. 상태 코드 기반 판정 ---
print("\n[판정]")
if resp.status_code == 200:
    print("  인증키 유효. 이번 호출이 정상 처리되었다 -> 사용량이 남아 있다(최소 1건 이상 호출 가능).")
    try:
        body = resp.json()
        feats = body.get("features", [])
        if feats:
            props = feats[0].get("properties", {})
            print(f"  응답 검증: feature {len(feats)}개, "
                  f"거리 {props.get('totalDistance', 0)}m, 시간 {props.get('totalTime', 0)}s")
    except json.JSONDecodeError:
        print("  (본문 JSON 파싱 실패 — 상태 코드만으로 정상 판정)")
elif resp.status_code == 429:
    print("  HTTP 429 -> 사용량 한도 소진 또는 순간 호출 제한. 아래 본문 code를 확인한다.")
    print("  code가 GW.QUOTA_LIMIT이면 할당량 초과이며, 리셋 시점까지 대기하거나 요금제를 상향해야 한다.")
    try:
        print(f"  본문: {json.dumps(resp.json(), ensure_ascii=False)[:400]}")
    except json.JSONDecodeError:
        print(f"  본문: {resp.text[:400]}")
elif resp.status_code in (401, 403):
    print("  인증 실패(401/403) -> 키가 유효하지 않거나 해당 상품(보행자 경로 안내) 이용 권한이 없다.")
    print(f"  본문: {resp.text[:400]}")
else:
    print(f"  예외 상태 {resp.status_code}. 본문을 확인한다.")
    print(f"  본문: {resp.text[:400]}")

print("=" * 64)


## 결과 해석

| HTTP 상태 | 의미 | 사용량 관점 |
| --- | --- | --- |
| 200 | 정상 응답 | 키 유효, 사용량 잔여(최소 1건 이상 호출 가능) |
| 429 | 요청량 초과 | 본문 `code`가 `GW.QUOTA_LIMIT`이면 할당량 소진 |
| 401 / 403 | 인증·권한 오류 | 키 오류 또는 상품 이용 권한 없음 (사용량과 무관) |

### 정확한 잔여 건수 확인

응답 본문에는 잔여 호출 건수가 표준 필드로 포함되지 않는다. 위 셀의 `[사용량 관련 응답 헤더]`에
`*-quota-remaining` 형태 값이 출력되면 그 값이 잔여 건수이며, 출력되지 않으면 다음 경로에서 확인한다.

- SK Open API 콘솔: 홈 > 대시보드 > 요금 조회 > 사용상품관리
- 보행자 경로 안내 상품/요금 페이지: https://openapi.sk.com/products/calc?svcSeq=4&menuSeq=5

즉 코드만으로 잔여 건수를 항상 얻을 수는 없으며, 호출 1건으로 확실히 판정되는 것은 **키 유효성**과
**한도 소진 여부(200 대 429)**이다. 정확한 남은 수치는 게이트웨이가 헤더로 노출할 때만 코드로 확보된다.
